## Imports

In [1]:
import IPython.display as ipd
import json
import numpy as np
import matplotlib.pyplot as plt
import random
import soundfile as sf
import pandas as pd

import tensorflow as tf
import keras

2025-09-16 23:35:41.896620: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 23:35:41.908722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758076541.921462   81030 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758076541.925034   81030 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758076541.934525   81030 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Carregando o modelo salvo

In [2]:
# Carregando o modelo salvo
model = keras.models.load_model('model_conv_1_sec_v1_0.keras')

# Carregando o scaler_y
import joblib
scaler_y = joblib.load('scaler_y_conv_1_sec_v1_0.save')

I0000 00:00:1758076543.691934   81030 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2756 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


## Carregando o dataset

In [3]:
## Carregando o dataset (lendo todos os arquivos .wav da pasta 'nsynth-test/audio')
import os
base_path = 'nsynth-test/audio'
file_list = [f for f in os.listdir(base_path) if f.endswith('.wav')]
samples = []
for file_name in file_list:
    signal = sf.read(os.path.join(base_path, file_name))
    samples.append(signal)

samples = pd.DataFrame(samples)
samples.head()

,0,1
0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16000
1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16000
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, -9.15527343...",16000
3,"[0.0, 0.0, 9.1552734375e-05, 0.000244140625, 0...",16000
4,"[0.0, 0.0, 0.0, 0.0, 3.0517578125e-05, -3.0517...",16000


In [4]:
samples = samples.drop(columns=[1])
samples.head()

,0
0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, -9.15527343..."
3,"[0.0, 0.0, 9.1552734375e-05, 0.000244140625, 0..."
4,"[0.0, 0.0, 0.0, 0.0, 3.0517578125e-05, -3.0517..."


## Ajustando o formato do dataset

In [5]:
print(samples.shape)
x = np.array(samples[0].values.tolist())
print(x.shape)

(4096, 1)
(4096, 64000)


In [6]:
x = x.reshape((x.shape[0], x.shape[1], 1))
print(x.shape)

print(x[2])

(4096, 64000, 1)
[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]


## Chamando o modelo para predição do dataset ajustado

In [7]:
## Chamando o modelo para predição do dataset ajustado
y_pred_norm = model.predict(x)
y_pred = scaler_y.inverse_transform(y_pred_norm)

I0000 00:00:1758076547.827979   81230 service.cc:152] XLA service 0x7f2a600052d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758076547.827999   81230 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2025-09-16 23:35:47.834185: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1758076547.857493   81230 cuda_dnn.cc:529] Loaded cuDNN version 90300


 10/128 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step

I0000 00:00:1758076548.743715   81230 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step


In [8]:
y_pred = pd.DataFrame(y_pred)
y_pred.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,1011.028870,1.572172,0.960548,1.743033,1.263678,0.831520,0.999319,1.375064,0.869788,1.378766,3.283947,0.726642,0.175128,0.415133,0.308988,0.514696
1,2167.062500,0.832551,0.514938,0.972516,0.560380,0.757105,0.457506,0.971913,0.602468,0.938196,1.771237,0.729750,0.093724,0.399406,0.191894,0.433087
2,1006.325256,1.245456,0.450953,1.499973,0.933158,0.904511,0.992478,1.495545,0.868119,1.612638,2.627724,0.711178,0.456967,0.235115,0.431532,0.898837
3,1130.110962,1.302395,0.759611,1.273861,0.887774,1.228408,0.971000,1.366397,0.797465,1.435779,2.521910,0.709827,0.252645,0.347197,0.472352,0.656200
4,822.036194,1.569117,0.971261,1.635816,1.153657,1.190513,1.110883,1.515270,0.938270,1.531909,3.269840,0.716137,0.217910,0.389767,0.416842,0.561215


In [9]:
## Escrevendo os parâmetros preditos em arquivo JSON
params_list = y_pred.to_dict(orient="records")
with open("nsynth-pred/_params_pred.json", "w") as f:
    json.dump(params_list, f, indent=4)

# Ressintetizando o nsynth

## Chamando o sintetizador para cada parâmetro predito pelo modelo

In [10]:
# Convertendo o nome das colunas para os nomes dos parâmetros do sintetizador
column_names = ['frequencia_base', 'frequency1', 'beta2',
       'frequency2', 'beta3', 'frequency3',
       'beta4', 'frequency4', 'beta5',
       'frequency5', 'beta_carrier', 'amplitude_carrier', 'attack', 'decay',
       'sustain', 'release']
y_pred.columns = column_names
y_pred.head()

,frequencia_base,frequency1,beta2,frequency2,beta3,frequency3,beta4,frequency4,beta5,frequency5,beta_carrier,amplitude_carrier,attack,decay,sustain,release
0,1011.028870,1.572172,0.960548,1.743033,1.263678,0.831520,0.999319,1.375064,0.869788,1.378766,3.283947,0.726642,0.175128,0.415133,0.308988,0.514696
1,2167.062500,0.832551,0.514938,0.972516,0.560380,0.757105,0.457506,0.971913,0.602468,0.938196,1.771237,0.729750,0.093724,0.399406,0.191894,0.433087
2,1006.325256,1.245456,0.450953,1.499973,0.933158,0.904511,0.992478,1.495545,0.868119,1.612638,2.627724,0.711178,0.456967,0.235115,0.431532,0.898837
3,1130.110962,1.302395,0.759611,1.273861,0.887774,1.228408,0.971000,1.366397,0.797465,1.435779,2.521910,0.709827,0.252645,0.347197,0.472352,0.656200
4,822.036194,1.569117,0.971261,1.635816,1.153657,1.190513,1.110883,1.515270,0.938270,1.531909,3.269840,0.716137,0.217910,0.389767,0.416842,0.561215


In [11]:
# Liberando memória e GPU
import gc
gc.collect()
keras.backend.clear_session()

In [12]:
## Chamando o sintetizador para cada parâmetro predito pelo modelo
from fm_synth2 import FMSynth

# Iterando sobre todas as predições
for i in range(y_pred.shape[0]):
    # Extraindo os parâmetros preditos
    params = y_pred.iloc[i].to_dict()
    
    # # Criando uma instância do sintetizador FM
    fm_synth = FMSynth(
        amplitude1=1.0,
        frequency1=params['frequency1'],
        beta2=params['beta2'],
        amplitude2=1.0,
        frequency2=params['frequency2'],
        beta3=params['beta3'],
        amplitude3=1.0,
        frequency3=params['frequency3'],
        beta4=params['beta4'],
        amplitude4=1.0,
        frequency4=params['frequency4'],
        beta5=params['beta5'],
        amplitude5=1.0,
        frequency5=params['frequency5'],
        beta_carrier=params['beta_carrier'],
        amplitude_carrier=params['amplitude_carrier'],
        attack=params['attack'],
        decay=params['decay'],
        sustain=params['sustain'],
        release=params['release'],
    )
    signal = fm_synth.synth_alg_series3_parallel2(4, params["frequencia_base"])

    # Normalizando o pico
    peak = np.max(np.abs(signal))
    if peak > 0:
        signal = 0.891 * signal / peak

    # Salvando o áudio em arquivo
    sf.write(f"nsynth-pred/{file_list[i]}", signal, 16000)
